# Comparison Summary — Day vs Week Approaches 🇬🇧 [🇮🇹](08_comparison_summary_IT.ipynb)

## Runner Injury Risk Prediction — Lovdal et al. (2021) Replication

This notebook consolidates findings from notebooks 04–07 into a unified
**day vs week comparison**. It does not retrain any model — it loads the
saved XGBoost models and recomputes metrics, SHAP values, and benchmark
comparisons dynamically.

### Contents

1. Setup
2. Data loading and predictions
3. Side-by-side metrics table
4. Comparison to paper benchmarks (Lovdal et al., 2021)
5. ROC and PR curve overlay
6. Feature importance comparison (SHAP)
7. Final recommendation
8. Limitations and future work
9. Summary

### Paper benchmarks

| Approach | AUC-ROC (paper) | Model |
|----------|-----------------|-------|
| Day | **0.724** | Bagged XGBoost (100 bags) |
| Week | **0.678** | Bagged XGBoost (100 bags) |

In [ ]:
import re
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns


# Ensure src/ is importable regardless of the notebook working directory
def _find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "src").exists() or (candidate / "pyproject.toml").exists():
            return candidate
    return start

PROJECT_ROOT = _find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import INJURY_COL, RANDOM_SEED
from src.data_loading import get_feature_columns
from src.interpretability.shap_analysis import (
    compute_shap_values,
    get_shap_importance_dict,
    get_top_features,
)
from src.modeling.evaluate import (
    compute_metrics,
    create_comparison_table,
    find_optimal_threshold,
    plot_pr_curves,
    plot_roc_curves,
)
from src.preprocessing.io import load_model, load_splits
from src.utils.logging_config import setup_logging
from src.utils.plotting import PALETTE, save_figure, set_style
from src.utils.reproducibility import set_global_seed

# --- Reproducibility & style ---
set_global_seed(RANDOM_SEED)
setup_logging()
set_style()

# --- Paper benchmarks ---
PAPER_BENCHMARK_DAY: float = 0.724
PAPER_BENCHMARK_WEEK: float = 0.678

---

## 2. Data Loading and Predictions

We load the tuned XGBoost models saved in notebooks 04 and 05, along with
the preprocessed train and test sets. For each approach we generate predicted
probabilities and find the optimal classification threshold on training set
predictions (maximizing F1), then apply it to the test set to avoid data leakage.

In [ ]:
# --- Day approach ---
day_model = load_model("day_best_model")
day_train, day_test = load_splits(prefix="day")
feature_cols_day = get_feature_columns(day_test)
X_train_day = day_train[feature_cols_day]
y_train_day = day_train[INJURY_COL]
X_test_day = day_test[feature_cols_day]
y_test_day = day_test[INJURY_COL]

# Threshold selected on train predictions to avoid test leakage
y_prob_train_day = day_model.predict_proba(X_train_day)[:, 1]
threshold_day = find_optimal_threshold(np.array(y_train_day), y_prob_train_day, metric="f1")
y_prob_day = day_model.predict_proba(X_test_day)[:, 1]
y_pred_day = (y_prob_day >= threshold_day).astype(int)

# --- Week approach ---
week_model = load_model("week_best_model")
week_train, week_test = load_splits(prefix="week")
feature_cols_week = get_feature_columns(week_test)
X_train_week = week_train[feature_cols_week]
y_train_week = week_train[INJURY_COL]
X_test_week = week_test[feature_cols_week]
y_test_week = week_test[INJURY_COL]

# Threshold selected on train predictions to avoid test leakage
y_prob_train_week = week_model.predict_proba(X_train_week)[:, 1]
threshold_week = find_optimal_threshold(np.array(y_train_week), y_prob_train_week, metric="f1")
y_prob_week = week_model.predict_proba(X_test_week)[:, 1]
y_pred_week = (y_prob_week >= threshold_week).astype(int)

# --- Summary ---
print("Day approach:")
print(f"  Test set: {X_test_day.shape[0]:,} rows x {X_test_day.shape[1]} features")
print(f"  Injury rate: {y_test_day.mean():.2%}")
print(f"  Train-selected threshold: {threshold_day:.2f}")
print(f"  Predicted positives: {y_pred_day.sum()} ({y_pred_day.mean():.2%})")

print("\nWeek approach:")
print(f"  Test set: {X_test_week.shape[0]:,} rows x {X_test_week.shape[1]} features")
print(f"  Injury rate: {y_test_week.mean():.2%}")
print(f"  Train-selected threshold: {threshold_week:.2f}")
print(f"  Predicted positives: {y_pred_week.sum()} ({y_pred_week.mean():.2%})")

---

## 3. Side-by-Side Metrics Table

Comprehensive comparison of the tuned XGBoost models on the held-out test set.
Both models are evaluated at their respective train-selected optimal thresholds
(maximizing F1 on training predictions, applied to test).
A delta column highlights which approach performs better on each metric.

In [ ]:
# Compute metrics for both approaches at optimal thresholds
day_metrics = compute_metrics(np.array(y_test_day), y_pred_day, y_prob_day)
week_metrics = compute_metrics(np.array(y_test_week), y_pred_week, y_prob_week)
day_xgb_label = "Day XGBoost"
week_xgb_label = "Week XGBoost"

# Build comparison table
comparison = create_comparison_table({
    day_xgb_label: day_metrics,
    week_xgb_label: week_metrics,
})

# Select key metrics and add delta (computed from unrounded dicts for accuracy)
key_metrics = ["auc_roc", "auc_pr", "recall", "precision", "f1", "brier_score"]
cmp_table = comparison[key_metrics].copy()
day_metric_values = pd.Series({metric: day_metrics[metric] for metric in key_metrics})
week_metric_values = pd.Series({metric: week_metrics[metric] for metric in key_metrics})
cmp_table.loc["Delta (Day − Week)"] = (day_metric_values - week_metric_values).round(4)

print(f"Test Set Metrics — Day vs Week (thresholds: day={threshold_day:.2f}, week={threshold_week:.2f}):\n")
print(cmp_table.to_string())

# Highlight winners
print("\n\nPer-metric winner:")
for metric in key_metrics:
    day_val = day_metrics[metric]
    week_val = week_metrics[metric]
    # Lower is better for Brier score
    if metric == "brier_score":
        winner = "Day" if day_val <= week_val else "Week"
    else:
        winner = "Day" if day_val >= week_val else "Week"
    print(f"  {metric:>15s}: {winner} ({day_val:.4f} vs {week_val:.4f})")

In [ ]:
# Grouped bar chart: day vs week on key metrics
display_metrics = ["auc_roc", "auc_pr", "recall", "precision", "f1", "brier_score"]
labels = ["AUC-ROC", "AUC-PR", "Recall", "Precision", "F1", "Brier Score"]

day_vals = [day_metrics[m] for m in display_metrics]
week_vals = [week_metrics[m] for m in display_metrics]

x = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
bars_day = ax.bar(x - width / 2, day_vals, width, label="Day", color=PALETTE["primary"], alpha=0.85)
bars_week = ax.bar(x + width / 2, week_vals, width, label="Week", color=PALETTE["secondary"], alpha=0.85)

# Value labels
for bars in [bars_day, bars_week]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                f"{bar.get_height():.3f}", ha="center", fontsize=9, fontweight="bold")

ax.set_ylabel("Score")
ax.set_title("Day vs Week — Test Set Metrics Comparison", fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylim(0, 1.05)
ax.legend(fontsize=11)
sns.despine()

save_figure(fig, "08_metrics_comparison_day_vs_week", subdir="comparison", close=False)
plt.show()
plt.close(fig)

---

## 4. Comparison to Paper Benchmarks

Lovdal et al. (2021) reported AUC-ROC of **0.724** (day) and **0.678** (week)
using a **bagged XGBoost ensemble** (100 bags). Our results use a single tuned
XGBoost model — a simpler architecture that is expected to produce somewhat
lower AUC-ROC.

In [ ]:
# Benchmark comparison table
our_day_auc = day_metrics["auc_roc"]
our_week_auc = week_metrics["auc_roc"]
our_xgb_label = "Ours (single XGBoost)"
paper_xgb_label = "Paper (bagged XGBoost)"

benchmark_data = {
    "Approach": ["Day", "Day", "Week", "Week"],
    "Source": [our_xgb_label, paper_xgb_label, our_xgb_label, paper_xgb_label],
    "AUC-ROC": [our_day_auc, PAPER_BENCHMARK_DAY, our_week_auc, PAPER_BENCHMARK_WEEK],
}

benchmark_df = pd.DataFrame(benchmark_data)
benchmark_df["Delta vs Paper"] = benchmark_df.apply(
    lambda row: row["AUC-ROC"] - (PAPER_BENCHMARK_DAY if row["Approach"] == "Day" else PAPER_BENCHMARK_WEEK),
    axis=1,
)
benchmark_df["% of Paper"] = benchmark_df.apply(
    lambda row: row["AUC-ROC"] / (PAPER_BENCHMARK_DAY if row["Approach"] == "Day" else PAPER_BENCHMARK_WEEK) * 100,
    axis=1,
)

print("Paper Benchmark Comparison:\n")
print(benchmark_df.to_string(index=False, float_format="%.4f"))

In [ ]:
# Bar chart: our AUC-ROC vs paper benchmarks
approaches = ["Day", "Week"]
our_aucs = [our_day_auc, our_week_auc]
paper_aucs = [PAPER_BENCHMARK_DAY, PAPER_BENCHMARK_WEEK]

x = np.arange(len(approaches))
width = 0.3

fig, ax = plt.subplots(figsize=(8, 5))
bars_ours = ax.bar(x - width / 2, our_aucs, width, label="Ours (single XGBoost)",
                   color=PALETTE["primary"], alpha=0.85)
bars_paper = ax.bar(x + width / 2, paper_aucs, width, label="Paper (bagged XGBoost)",
                    color=PALETTE["neutral"], alpha=0.7, hatch="//", edgecolor="white")

# Value labels
for bars in [bars_ours, bars_paper]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                f"{bar.get_height():.3f}", ha="center", fontsize=11, fontweight="bold")

ax.set_ylabel("AUC-ROC")
ax.set_title("AUC-ROC — Our Results vs Paper Benchmarks", fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(approaches)
ax.set_ylim(0, 1.0)
ax.legend(fontsize=10)
sns.despine()

save_figure(fig, "08_paper_benchmark_comparison", subdir="comparison", close=False)
plt.show()
plt.close(fig)

### Why our results differ from the paper

Several methodological differences explain the AUC-ROC gap:

1. **No bagging ensemble**: the paper used a 100-bag XGBoost ensemble that
   averages predictions across 100 independently trained models, reducing
   variance. We used a single tuned XGBoost — a deliberate simplicity choice.

2. **Stricter athlete separation (ADR-006)**: we enforce `GroupKFold` and
   `GroupShuffleSplit` by Athlete ID throughout. The paper's exact splitting
   strategy may differ, and stricter separation typically reduces apparent
   performance (no information leakage across athletes).

3. **Hyperparameter tuning budget**: we used `RandomizedSearchCV` with 30
   iterations for local feasibility. The paper likely used a more exhaustive
   search or domain-informed defaults.

4. **Sentinel handling (ADR-007)**: we replaced sentinel values (-0.01) with
   0.0 to represent rest days. Different treatments of these values affect
   the learned feature thresholds.

5. **Week target binarization (ADR-002)**: we binarized at threshold 0.5,
   consistent with the paper, but implementation details in the continuous-to-binary
   conversion may introduce small differences.

Despite these differences, our results are within a reasonable range of the
paper benchmarks, validating the replication's core methodology.

---

## 5. ROC and PR Curve Overlay

Overlaying the day and week ROC/PR curves on the same axes allows direct
visual comparison of discriminative performance across all classification
thresholds.

In [ ]:
# Combined ROC curves — day and week on same axes
combined_predictions = {
    day_xgb_label: (np.array(y_test_day), y_prob_day),
    week_xgb_label: (np.array(y_test_week), y_prob_week),
}

fig_roc = plot_roc_curves(combined_predictions, save_path=None)
fig_roc.axes[0].annotate(
    f"Paper benchmarks: Day={PAPER_BENCHMARK_DAY}, Week={PAPER_BENCHMARK_WEEK}",
    xy=(0.45, 0.05), xycoords="axes fraction", fontsize=10, fontstyle="italic", color=PALETTE["negative"],
)
save_figure(fig_roc, "08_roc_curves_combined", subdir="comparison", close=False)
plt.show()
plt.close(fig_roc)

In [ ]:
# Combined PR curves — day and week on same axes
# Note: PR curves are prevalence-dependent; the two test sets may have
# slightly different positive rates, so compare shapes with caution.
day_pos_rate = float(np.mean(combined_predictions["Day XGBoost"][0]))
week_pos_rate = float(np.mean(combined_predictions["Week XGBoost"][0]))

fig_pr = plot_pr_curves(combined_predictions, save_path=None)
fig_pr.axes[0].annotate(
    f"Positive rate — Day: {day_pos_rate:.3f}, Week: {week_pos_rate:.3f}\n"
    "PR curves are prevalence-dependent; compare with caution.",
    xy=(0.02, 0.02), xycoords="axes fraction", fontsize=9, fontstyle="italic",
    color=PALETTE["negative"],
    bbox={"boxstyle": "round,pad=0.3", "facecolor": "white", "alpha": 0.8, "edgecolor": "none"},
)
save_figure(fig_pr, "08_pr_curves_combined", subdir="comparison", close=False)
plt.show()
plt.close(fig_pr)

---

## 6. Feature Importance Comparison (SHAP)

SHAP-based feature importance reveals which training features drive predictions
in each temporal approach. Since the day and week approaches use different
feature sets (7 days x 10 features vs 3 weeks x 22 features + 3 ratios),
we compare at two levels:

1. **Raw feature level**: top features per approach (different feature names)
2. **Concept level**: stripping temporal prefixes to identify which base
   training metrics (volume, intensity, recovery, etc.) are consistently
   important regardless of temporal granularity

In [ ]:
# Compute SHAP values for both approaches
shap_values_day = compute_shap_values(day_model, X_test_day)
shap_values_week = compute_shap_values(week_model, X_test_week)

# Top 10 features per approach
top10_day = get_top_features(shap_values_day, n=10)
top10_week = get_top_features(shap_values_week, n=10)

print("Top 10 features by mean |SHAP value|:\n")
print(f"  {'Day Approach':<35s}  {'Week Approach':<35s}")
print(f"  {'─' * 35}  {'─' * 35}")
for i in range(10):
    d = top10_day[i] if i < len(top10_day) else ""
    w = top10_week[i] if i < len(top10_week) else ""
    print(f"  {i + 1:>2d}. {d:<32s}  {i + 1:>2d}. {w:<32s}")

In [ ]:
# Two-panel chart: top 15 SHAP features per approach
shap_imp_day = get_shap_importance_dict(shap_values_day)
shap_imp_week = get_shap_importance_dict(shap_values_week)

top_n = 15

# Sort by importance, take top N
day_sorted = sorted(shap_imp_day.items(), key=lambda x: x[1], reverse=True)[:top_n]
week_sorted = sorted(shap_imp_week.items(), key=lambda x: x[1], reverse=True)[:top_n]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# Day panel
day_names = [x[0] for x in reversed(day_sorted)]
day_values = [x[1] for x in reversed(day_sorted)]
ax1.barh(day_names, day_values, color=PALETTE["primary"], alpha=0.85)
ax1.set_xlabel("Mean |SHAP value|")
ax1.set_title("Day Approach — Top 15 Features", fontweight="bold")

# Week panel
week_names = [x[0] for x in reversed(week_sorted)]
week_values = [x[1] for x in reversed(week_sorted)]
ax2.barh(week_names, week_values, color=PALETTE["secondary"], alpha=0.85)
ax2.set_xlabel("Mean |SHAP value|")
ax2.set_title("Week Approach — Top 15 Features", fontweight="bold")

fig.suptitle("SHAP Feature Importance — Day vs Week", fontweight="bold", fontsize=14, y=1.02)
fig.tight_layout()

save_figure(fig, "08_shap_importance_day_vs_week", subdir="comparison", close=False)
plt.show()
plt.close(fig)

In [ ]:
# Concept-level overlap: strip temporal prefix and aggregate by base feature type
def extract_base_feature(feature_name: str) -> str:
    """Strip temporal prefix (day_N_ or week_N_) to get the base feature type."""
    return re.sub(r"^(day|week)_\d+_", "", feature_name)


def aggregate_importance_by_type(importance: dict[str, float]) -> dict[str, float]:
    """Sum SHAP importance across temporal windows for each base feature type."""
    agg: dict[str, float] = {}
    for feat, val in importance.items():
        base = extract_base_feature(feat)
        agg[base] = agg.get(base, 0.0) + val
    return agg


day_by_type = aggregate_importance_by_type(shap_imp_day)
week_by_type = aggregate_importance_by_type(shap_imp_week)

# Common base feature types
common_types = sorted(set(day_by_type) & set(week_by_type))

print(f"Day unique feature types: {len(day_by_type)}")
print(f"Week unique feature types: {len(week_by_type)}")
print(f"Common feature types: {len(common_types)}")

if common_types:
    # Normalize to percentages for fair comparison
    day_total = sum(day_by_type[t] for t in common_types)
    week_total = sum(week_by_type[t] for t in common_types)

    day_pct = {t: day_by_type[t] / day_total * 100 for t in common_types}
    week_pct = {t: week_by_type[t] / week_total * 100 for t in common_types}

    # Sort by average importance
    sorted_types = sorted(common_types, key=lambda t: (day_pct[t] + week_pct[t]) / 2, reverse=True)

    print("\nCommon feature types ranked by average importance:\n")
    print(f"  {'Feature type':<35s}  {'Day %':>8s}  {'Week %':>8s}")
    print(f"  {'─' * 35}  {'─' * 8}  {'─' * 8}")
    for t in sorted_types:
        print(f"  {t:<35s}  {day_pct[t]:>7.1f}%  {week_pct[t]:>7.1f}%")

---

## 7. Final Recommendation

In [ ]:
# Programmatic summary: which approach wins on each dimension
print("=" * 60)
print("RECOMMENDATION SUMMARY")
print("=" * 60)

# Performance comparison
metrics_to_compare = {
    "AUC-ROC": ("auc_roc", "higher"),
    "AUC-PR": ("auc_pr", "higher"),
    "Recall": ("recall", "higher"),
    "Precision": ("precision", "higher"),
    "F1": ("f1", "higher"),
    "Brier Score": ("brier_score", "lower"),
}

day_wins = 0
week_wins = 0

print("\n1. PERFORMANCE (test set, train-selected thresholds)")
print(f"   {'Metric':<15s}  {'Day':>8s}  {'Week':>8s}  {'Winner':>8s}")
print(f"   {'─' * 15}  {'─' * 8}  {'─' * 8}  {'─' * 8}")

for label, (key, direction) in metrics_to_compare.items():
    d = day_metrics[key]
    w = week_metrics[key]
    if direction == "higher":
        winner = "Day" if d >= w else "Week"
    else:
        winner = "Day" if d <= w else "Week"
    if winner == "Day":
        day_wins += 1
    else:
        week_wins += 1
    print(f"   {label:<15s}  {d:>8.4f}  {w:>8.4f}  {winner:>8s}")

print(f"\n   Score: Day {day_wins} — Week {week_wins}")

# Paper proximity
day_pct_paper = our_day_auc / PAPER_BENCHMARK_DAY * 100
week_pct_paper = our_week_auc / PAPER_BENCHMARK_WEEK * 100
print("\n2. PAPER BENCHMARK PROXIMITY")
print(f"   Day:  {our_day_auc:.4f} / {PAPER_BENCHMARK_DAY} = {day_pct_paper:.1f}% of paper")
print(f"   Week: {our_week_auc:.4f} / {PAPER_BENCHMARK_WEEK} = {week_pct_paper:.1f}% of paper")

print("\n3. INTERPRETABILITY (from notebook 06)")
print(f"   Day:  {len(feature_cols_day)} features (7 days x 10 metrics) — finer temporal resolution")
print(f"   Week: {len(feature_cols_week)} features (3 weeks x 22+ metrics) — includes relative km ratios")

print("\n4. FAIRNESS (from notebook 07)")
print("   Both approaches audited across 3 proxy grouping strategies.")
print("   Cross-comparison in reports/figures/fairness/07_day_vs_week_disparity_comparison.png")

print("\n" + "=" * 60)

### Recommendation

Based on the multi-dimensional comparison above:

**1. Discriminative performance**: the day approach consistently achieves
higher AUC-ROC and AUC-PR, consistent with the paper's findings. Daily
granularity captures acute load spikes that weekly aggregation smooths out.

**2. Clinical utility**: in injury prevention, **recall** (detecting actual
injuries) is more important than precision (avoiding false alarms). A missed
injury (false negative) has higher cost than an unnecessary caution flag
(false positive). The approach with higher recall at reasonable precision
should be preferred for deployment.

**3. Interpretability**: both approaches yield interpretable SHAP patterns.
The day approach reveals acute daily load effects; the week approach captures
longer-term load accumulation and acute-to-chronic ratios — both are
meaningful to sports scientists and coaches.

**4. Fairness**: proxy-group analysis (notebook 07) showed that both approaches
have similar fairness profiles. Neither approach is systematically more
equitable than the other across volume, injury history, and data density
groupings.

**5. Practical deployment**:
- **Day approach**: provides earlier warning (daily predictions), enables
  same-day training adjustments, higher temporal resolution
- **Week approach**: more stable predictions (less noise from single-day
  fluctuations), includes acute-to-chronic load ratios (a well-validated
  sports science concept), easier for coaches to act on weekly plans

**6. Overall**: both approaches are valid. The choice depends on the
operational context — daily monitoring systems favor the day approach,
while weekly planning workflows favor the week approach. For a general
injury risk prediction system, the **day approach** offers a slight edge
in discriminative performance and temporal resolution.

---

## 8. Limitations and Future Work

### Limitations

1. **No bagging ensemble**: the paper's 100-bag XGBoost achieves higher
   AUC-ROC through variance reduction. Our single-model architecture is
   simpler but less performant.

2. **Small athlete cohort**: 74 athletes total, with approximately 15 in
   the test set. Results may not generalize to larger or different
   populations (e.g., recreational runners, different sports).

3. **No external validation**: we evaluate on a held-out split of the same
   dataset used by Lovdal et al. True generalizability requires testing on
   an independent dataset.

4. **Severe class imbalance**: the day approach has only 1.36% positive
   samples. This makes calibration unreliable and metrics like precision
   and F1 highly sensitive to threshold choice.

5. **No temporal validation**: we use GroupKFold (ADR-006) instead of
   TimeSeriesSplit. While this prevents athlete leakage, it does not
   test the model's ability to predict future injuries from past data
   in a strictly temporal sense.

6. **No demographic fairness**: without age, sex, or other protected
   attributes, proxy-group analysis provides limited fairness assurance.
   Systematic biases across demographic groups cannot be detected or ruled out.

7. **Single dataset, single sport**: results are specific to elite Dutch
   runners (2012–2019). Transferability to other sports, levels, or
   monitoring systems is unknown.

### Future work

1. **Bagged XGBoost ensemble**: implement 100-bag architecture to close
   the gap with paper benchmarks and quantify variance reduction
2. **Deep learning approaches**: LSTM or Transformer models for time-series
   patterns, potentially capturing long-range dependencies
3. **Demographic data collection**: enable proper fairness auditing across
   protected attributes
4. **External validation**: test on independent runner populations
   (different countries, competition levels, monitoring systems)
5. **Prospective study design**: train on historical data (e.g., 2012–2017),
   validate on future seasons (2018–2019) to test temporal generalization
6. **Calibration improvement**: Platt scaling or isotonic regression to
   produce reliable injury probabilities (not just rankings)
7. **Interactive dashboard**: Looker Studio deployment for real-time
   monitoring by coaches and sports scientists (deferred Phase 8)

---

## 9. Summary

### Pipeline flow

```
data/processed/day_best_model.pkl + week_best_model.pkl
  → load_model()
data/processed/day_test.parquet + week_test.parquet
  → load_splits()
  → compute_metrics() [side-by-side comparison]
  → find_optimal_threshold() [per-approach F1 optimization]
  → plot_roc_curves() + plot_pr_curves() [overlaid curves]
  → compute_shap_values() [feature importance comparison]
  → Concept-level overlap analysis [base feature types]
  → All figures saved to reports/figures/comparison/
```

### Figures generated

All figures saved to `reports/figures/comparison/`:
- 1 side-by-side metrics bar chart (day vs week)
- 1 paper benchmark comparison chart
- 1 combined ROC curve overlay
- 1 combined PR curve overlay
- 1 two-panel SHAP importance chart

**Total: 5 figures**

### Key findings

1. **Day approach outperforms week approach** in discriminative metrics
   (AUC-ROC, AUC-PR), consistent with the paper's findings. Daily
   granularity captures acute load signals that weekly aggregation smooths.

2. **Both approaches fall below paper benchmarks**, primarily due to using
   a single XGBoost model vs. the paper's 100-bag ensemble, and stricter
   athlete-level separation (GroupKFold/GroupShuffleSplit).

3. **Feature importance is consistent at the concept level**: training
   volume and subjective markers (exertion, recovery) are important in
   both approaches, validating their role as injury risk indicators.

4. **Both approaches show similar fairness profiles** across proxy groups —
   neither is systematically more equitable.

5. **The choice between approaches depends on operational context**: day for
   real-time monitoring, week for planning workflows. The day approach has
   a slight overall edge.

6. **The replication validates the core methodology**: despite architectural
   simplifications, the same data patterns and model rankings emerge.